# NGC 4395 Drizzle3D Pipeline — Step-by-Step Tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WenkeRen/spxquery/blob/feat/drizzle3d/example/drizzle3d_demo/ngc4395_pipeline.ipynb)

This notebook demonstrates the full SPHEREx Drizzle3D pipeline:

1. **Config creation & YAML workflow** — build config, export to YAML, edit, reload
2. **Build output grids** — spatial WCS + spectral Z grid
3. **Query IRSA TAP** — find observations covering the target
4. **Download observations** — parallel FITS download (local) or pre-downloaded from Drive (Colab)
5. **Drizzle into 3D cube** — combine observations into a data cube
6. **Static visualization** — Z-collapsed coadd, spectra, individual slices
7. **Interactive Z-bin slider** — browse wavelength slices
8. **Interactive click-to-spectrum** — click any pixel to see its spectrum

**Colab:** Click the badge above to open in Colab. The first cell auto-installs dependencies and downloads D3 images from Google Drive.

**Local:**
```bash
pip install spxquery ipympl ipywidgets
# or with poetry:
poetry install --with notebooks
```

**Target:** NGC 4395 (RA = 186.4536, Dec = 33.5468), 30' x 30' region, Detector D3 (1.66-2.44 um)

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("[Colab] Installing spxquery + interactive dependencies ...")
    !pip install -q git+https://github.com/WenkeRen/spxquery.git@feat/drizzle3d
    !pip install -q ipympl ipywidgets gdown

    DEMO_DIR = Path("/content/ngc4395_demo")
    OUTPUT_DIR = DEMO_DIR / "output"
    YAML_PATH = DEMO_DIR / "ngc4395_config.yaml"

    # Download D3 images from Google Drive (shared folder)
    D3_DIR = OUTPUT_DIR / "images" / "D3"
    os.makedirs(D3_DIR, exist_ok=True)
    n_fits = len(list(D3_DIR.glob("*.fits")))
    if n_fits < 38:
        import gdown

        print(f"[Colab] Downloading D3 images from Google Drive (~2.6 GB, found {n_fits}/38) ...")
        gdown.download_folder(
            url="https://drive.google.com/drive/folders/1BeQ6FvhhgiZwjrxPBnHxI5rZ6Ps3oW47",
            output=str(D3_DIR),
            quiet=False,
            remaining_ok=True,
        )
        n_fits = len(list(D3_DIR.glob("*.fits")))
        print(f"[Colab] Downloaded {n_fits} D3 images")
    else:
        print(f"[Colab] D3 data already present ({n_fits} files)")
else:
    import spxquery

    _root = Path(spxquery.__file__).resolve().parent.parent.parent
    DEMO_DIR = _root / "example" / "drizzle3d_demo"
    OUTPUT_DIR = DEMO_DIR / "output"
    YAML_PATH = DEMO_DIR / "ngc4395_config.yaml"


In [ ]:
# ── Matplotlib backend ─────────────────────────────────────────
try:
    get_ipython().run_line_magic("matplotlib", "widget")
except ValueError:
    # Colab: ipympl backend unavailable until runtime restart
    get_ipython().run_line_magic("matplotlib", "inline")
    print("Note: ipympl unavailable. Using inline backend.")
    print("      For full interactivity: Runtime > Restart session, then re-run.")

import logging

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact

logging.basicConfig(level=logging.INFO, format="%(name)s - %(levelname)s - %(message)s")

from spxquery.drizzle3d import Drizzle3DConfig

print(f"Demo dir  : {DEMO_DIR}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"YAML path : {YAML_PATH}")
if IN_COLAB:
    d3_files = list(D3_DIR.glob("*.fits")) if D3_DIR.exists() else []
    print(f"D3 images: {len(d3_files)} files ready")

---
## Step 1: Configuration & YAML Workflow

We create a `Drizzle3DConfig` with all parameters for the NGC 4395 target, then:

1. **Export** to a YAML file for reproducibility
2. **Inspect** the YAML content
3. **Edit** the YAML (manually, or re-run with different values)
4. **Reload** the config from YAML

This pattern lets you save the config alongside your output, making every run reproducible.

In [ ]:
# Create configuration for NGC 4395
config = Drizzle3DConfig(
    center_ra=186.4536,
    center_dec=33.5468,
    width=30.0,  # arcmin
    height=30.0,
    detector=3,  # D3 only (1.66–2.44 μm)
    xy_shrink=0.8,  # shrink input droplets to 80% of native size
    output_dir=OUTPUT_DIR,
    subtract_zodi=True,
    overwrite=True,
)

# Print a summary of the config
print("=== Drizzle3DConfig ===")
print(f"  Target   : NGC 4395 @ ({config.center_ra}, {config.center_dec})")
print(f"  Region   : {config.width}' × {config.height}'")
print(f"  Detector : D{config.detector}")
print(f"  Grid     : {config.output_nx()} × {config.output_ny()} pix")
print(f"  Pixscale : {config.effective_pixscale():.3f} pix")
print(f"  XY shrink: {config.xy_shrink}")
print(f"  Z  shrink: {config.effective_z_shrink()}")
print()

# Export to YAML
config.to_yaml_file(YAML_PATH)
print(f"Config exported to: {YAML_PATH}\n")

# Display the YAML content
print("--- YAML Content ---")
print(YAML_PATH.read_text())

### Edit the YAML (optional)

You can open `ngc4395_config.yaml` in a text editor and change any value. For example:

- Change `xy_shrink` to `1.0` for full-pixel droplets
- Set `mjd_range` to `[60600, 60800]` to restrict the time range
- Change `detector` to `0` to process all detectors
- Toggle `subtract_zodi` to `false` to skip zodiacal background subtraction
- Set `data_mirror` to use a local mirror instead of downloading (see below)

#### Local mirror setup

`data_mirror` points to the directory that contains data release folders (`qr2/`, `qr3/`, etc.).
The path mapping strips the IRSA IBE prefix and preserves everything after the release mark:

```
IRSA URL : https://irsa.ipac.caltech.edu/ibe/data/spherex/qr2/level2/.../xxx.fits
                                                                        ↑
                                                            data release mark
Strip    : https://irsa.ipac.caltech.edu/ibe/data/spherex
Remainder: qr2/level2/.../xxx.fits
Mirror   : data_mirror/qr2/level2/.../xxx.fits
```

Example YAML entry:
```yaml
data_mirror: /data/spherex_mirror
```

Expected directory tree:
```
/data/spherex_mirror/
├── qr2/
│   └── level2/
│       └── 2025W20_2D/
│           └── .../xxx.fits
└── qr3/          ← future releases work automatically
    └── level2/
        └── ...
```

When you are ready, run the next cell to reload the config.

In [ ]:
# Reload config from YAML — picks up any edits you made
config = Drizzle3DConfig.from_yaml_file(YAML_PATH)

print("=== Loaded Config (from YAML) ===")
print(f"  center_ra    = {config.center_ra}")
print(f"  center_dec   = {config.center_dec}")
print(f"  width        = {config.width}'")
print(f"  height       = {config.height}'")
print(f"  detector     = D{config.detector}")
print(f"  xy_shrink    = {config.xy_shrink}")
print(f"  z_shrink     = {config.z_shrink}")
print(f"  xy_oversample= {config.xy_oversample}")
print(f"  z_oversample = {config.z_oversample}")
print(f"  subtract_zodi= {config.subtract_zodi}")
print(f"  exclude_flags= {config.exclude_flags}")
print(f"  output_dir   = {config.output_dir}")
print(f"  Grid         : {config.output_nx()} × {config.output_ny()} pix, {config.effective_pixscale():.3f} pix")

---
## Step 2: Build Output Grids (WCS + Z axis)

The output grid has two independent components:

- **Spatial (WCS)**: a gnomonic (TAN) projection centered on the target. Pixel scale = 6.15" / oversample.
- **Spectral (Z)**: wavelength bins from the detector's spectral channels (17 native subchannels per detector).

In [ ]:
from spxquery.drizzle3d.grid import build_output_wcs
from spxquery.drizzle3d.spectral import build_z_grid

output_wcs = build_output_wcs(config)
zgrid = build_z_grid(config.detector, config.z_oversample, config.z_lambda_edges)

print("=== Spatial WCS ===")
print(f"  Projection: TAN (gnomonic)")
print(f"  CRVAL  : ({output_wcs.wcs.crval[0]:.4f}, {output_wcs.wcs.crval[1]:.4f}) deg")
print(f"  CRPIX  : ({output_wcs.wcs.crpix[0]:.1f}, {output_wcs.wcs.crpix[1]:.1f}) pix")
print(f"  CDELT  : ({output_wcs.wcs.cdelt[0]:.2e}, {output_wcs.wcs.cdelt[1]:.2e}) deg")
print(f"  Shape  : ({config.output_ny()}, {config.output_nx()}) pixels")
print()
print("=== Spectral Z Grid ===")
print(f"  N bins : {zgrid.n_z}")
print(f"  Lambda : [{zgrid.centers[0]:.4f}, {zgrid.centers[-1]:.4f}] μm")
print(f"  dLambda: {zgrid.widths[0]:.4f} – {zgrid.widths[-1]:.4f} μm")
print(f"  R_eff  : {(zgrid.centers / zgrid.widths).mean():.0f}")

---
## Step 3: Query IRSA TAP

Query the IRSA archive for SPHEREx observations that cover the target region.
The ADQL query uses `CONTAINS(POINT, poly)` to find observations whose
footprint overlaps the target position.

In [ ]:
from spxquery.drizzle3d.query import query_observations

try:
    obs_by_det = query_observations(config)
    n_total = sum(len(v) for v in obs_by_det.values())
    print(f"Found {n_total} observations across {len(obs_by_det)} detector(s)")
    for det, obs_list in sorted(obs_by_det.items()):
        print(f"  D{det}: {len(obs_list)} observations")
except Exception as e:
    print(f"TAP query failed: {e}")
    obs_by_det = {}

if not obs_by_det:
    print("No data found — cannot continue pipeline.")

---
## Step 4: Resolve Data Files

Resolve FITS file paths for the queried observations. Two modes:

- **Download mode** (default): downloads FITS files from IRSA via HTTP.
  Already-downloaded files are skipped if `skip_existing=True`.
- **Mirror mode** (`data_mirror` set in config): resolves paths from a local
  mirror by mapping the IRSA URL to `data_mirror/qr2/level2/.../*.fits`.
  Verifies each file exists. No network access.

In [ ]:
from spxquery.drizzle3d.query import download_observations

all_fits = {}
for det, obs_list in sorted(obs_by_det.items()):
    paths = download_observations(
        obs_list,
        output_dir=OUTPUT_DIR,
        max_workers=config.download_workers,
        skip_existing=config.skip_existing,
        data_mirror=config.data_mirror,
    )
    all_fits[det] = paths
    print(f"  D{det}: resolved {len(paths)}/{len(obs_list)} files")

---
## Step 5: Drizzle into 3D Cube

This is the core step. For each detector:

1. Build the spectral Z grid for that detector
2. Create a `DrizzleCube` accumulator (3D arrays: flux, weight, variance, masks)
3. Loop over input FITS files:
   - Read image, variance, flags, and both spatial + spectral WCS
   - Compute spatial mapping (bilinear weight distribution)
   - Compute spectral overlap (vectorized top-hat kernel)
   - Accumulate into the cube via `np.add.at`
4. Finalize masks and save the output cube

The `drizzle_detector()` function encapsulates the per-detector loop.

In [ ]:
from spxquery.drizzle3d.pipeline import drizzle_detector

for det, fits_paths in sorted(all_fits.items()):
    if not fits_paths:
        continue

    outpath = drizzle_detector(fits_paths, config, det, output_wcs)
    if outpath:
        print(f"  D{det} cube saved: {outpath}")

# List output files
cube_files = sorted(OUTPUT_DIR.glob("drizzle_D*.fits"))
print(f"\nOutput cubes: {[f.name for f in cube_files]}")

---
## Step 6: Static Visualization

Load the output cube and create a multi-panel figure:

- **Z-collapsed coadd** with peak pixel marker
- **Coverage map** (median input count per voxel)
- **Spectrum** at the peak pixel with error band
- **Three individual wavelength slices** (first, middle, last Z bins)

In [ ]:
from spxquery.drizzle3d.io import load_cube

cube_files = sorted(OUTPUT_DIR.glob("drizzle_D*.fits"))
if not cube_files:
    print("No output cubes found.")
else:
    for cube_path in cube_files:
        data = load_cube(cube_path)
        sci = data["sci"]
        var = data["variance"]
        wl = data["wavelength"]
        hdr = data["header"]
        n_z, n_y, n_x = sci.shape
        lam_arr = wl["LAMBDA"]

        # Find peak pixel (galaxy nucleus)
        coadd = np.nansum(sci, axis=0)
        peak_y, peak_x = np.unravel_index(np.nanargmax(coadd), coadd.shape)

        fig, axes = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)

        # Row 1, Col 0: Z-collapsed coadd
        with np.errstate(invalid="ignore"):
            coadd_pos = np.where(coadd > 0, coadd, np.nan)
        vmin, vmax = np.nanpercentile(coadd_pos, 5), np.nanpercentile(coadd_pos, 99.5)
        im0 = axes[0, 0].imshow(coadd, origin="lower", cmap="inferno", vmin=vmin, vmax=vmax)
        axes[0, 0].plot(peak_x, peak_y, "c+", ms=12, mew=1.5)
        axes[0, 0].set_title(f"D{hdr['DETECTOR']} Z-collapsed (peak at {peak_x},{peak_y})")
        axes[0, 0].set_xlabel("X pixel")
        axes[0, 0].set_ylabel("Y pixel")
        plt.colorbar(im0, ax=axes[0, 0], label=r"$\Sigma$ flux [MJy/sr]")

        # Row 1, Col 1: Coverage map
        cnt = data.get("count")
        if cnt is not None:
            im1 = axes[0, 1].imshow(np.median(cnt, axis=0), origin="lower", cmap="hot")
            axes[0, 1].set_title("Median input count per voxel")
            plt.colorbar(im1, ax=axes[0, 1], label="N pixels")
        else:
            coverage = np.sum(np.isfinite(sci), axis=0)
            im1 = axes[0, 1].imshow(coverage, origin="lower", cmap="hot")
            axes[0, 1].set_title("Coverage (N Z-planes)")
            plt.colorbar(im1, ax=axes[0, 1], label="N planes")
        axes[0, 1].set_xlabel("X pixel")

        # Row 1, Col 2: Spectrum at peak pixel
        spec_peak = sci[:, peak_y, peak_x]
        var_peak = var[:, peak_y, peak_x]
        valid = np.isfinite(spec_peak)
        if np.any(valid):
            axes[0, 2].errorbar(
                lam_arr[valid],
                spec_peak[valid],
                yerr=np.sqrt(var_peak[valid]),
                fmt="o-",
                markersize=3,
                capsize=2,
                color="C1",
            )
        axes[0, 2].set_xlabel(r"$\lambda$ [$\mu$m]")
        axes[0, 2].set_ylabel("Flux [MJy/sr]")
        axes[0, 2].set_title(f"Spectrum at peak ({peak_x},{peak_y})")
        axes[0, 2].grid(True, alpha=0.3)

        # Row 2: Individual Z-bin images
        z_indices = [0, n_z // 2, n_z - 1]
        for panel_idx, zi in enumerate(z_indices):
            img = sci[zi]
            lam = lam_arr[zi]
            v1, v2 = np.nanpercentile(img, [1, 99.5])
            ax = axes[1, panel_idx]
            im = ax.imshow(img, origin="lower", cmap="inferno", vmin=v1, vmax=v2)
            ax.plot(peak_x, peak_y, "c+", ms=10, mew=1.5)
            ax.set_title(f"Z-bin {zi}: $\lambda$={lam:.3f} $\mu$m")
            ax.set_xlabel("X pixel")
            if panel_idx == 0:
                ax.set_ylabel("Y pixel")
            plt.colorbar(im, ax=ax, label="MJy/sr")

        fig.suptitle(
            f"NGC 4395 — Drizzle3D D{hdr['DETECTOR']} ({hdr['N_INPUTS']} inputs)",
            fontsize=13,
        )

---
## Step 7: Interactive Z-bin Browser

Drag the slider to browse different wavelength slices.
Color limits auto-adjust per slice using the [1%, 99.5%] percentile range.

In [ ]:
fig_z, ax_z = plt.subplots(constrained_layout=True)
im_z = ax_z.imshow(sci[0], origin="lower", cmap="inferno")
plt.colorbar(im_z, ax=ax_z, label="MJy/sr")
title_z = ax_z.set_title(f"Z-bin 0: $\lambda$ = {lam_arr[0]:.4f} $\mu$m")
ax_z.set_xlabel("X pixel")
ax_z.set_ylabel("Y pixel")


@interact(
    z_bin=widgets.IntSlider(
        min=0,
        max=n_z - 1,
        step=1,
        value=0,
        description="Z-bin",
        layout=widgets.Layout(width="500px"),
    )
)
def update_z(z_bin):
    img = sci[z_bin]
    v1, v2 = np.nanpercentile(img, [1, 99.5])
    im_z.set_data(img)
    im_z.set_clim(v1, v2)
    title_z.set_text(f"Z-bin {z_bin}: $\lambda$ = {lam_arr[z_bin]:.4f} $\mu$m")
    fig_z.canvas.draw_idle()

---
## Step 8: Interactive Click-to-Spectrum Viewer

The left panel shows a median-Z spatial map. **Left-click anywhere on the map**
and the right panel will display the full spectrum (with error band) at that pixel.

In [ ]:
fig_sv, (ax_sky, ax_spec) = plt.subplots(
    1, 2, figsize=(14, 5.5), gridspec_kw={"width_ratios": [1, 1]}, constrained_layout=True
)

# Median-Z spatial map
median_map = np.nanmedian(sci, axis=0)
v1, v2 = np.nanpercentile(median_map, [1, 99])
im_sv = ax_sky.imshow(median_map, origin="lower", cmap="inferno", vmin=v1, vmax=v2)
plt.colorbar(im_sv, ax=ax_sky, label="MJy/sr")
ax_sky.set_xlabel("X pixel")
ax_sky.set_ylabel("Y pixel")
ax_sky.set_title("Median flux — click to select pixel")

# Crosshair marker
(marker,) = ax_sky.plot([], [], "c+", ms=15, mew=2)

# Spectrum axes
(line_spec,) = ax_spec.plot([], [], "o-", color="C1", ms=4, lw=1)
ax_spec.set_xlabel(r"$\lambda$ [$\mu$m]")
ax_spec.set_ylabel("Flux [MJy/sr]")
ax_spec.set_title("Click on the map to show spectrum")
ax_spec.grid(True, alpha=0.3)

_fill = [None]


def _on_click(event):
    if event.inaxes != ax_sky or event.button != 1:
        return
    x, y = int(round(event.xdata)), int(round(event.ydata))
    if not (0 <= x < n_x and 0 <= y < n_y):
        return

    spec = sci[:, y, x].astype(np.float64)
    sigma = np.sqrt(var[:, y, x].astype(np.float64))
    valid = np.isfinite(spec)

    marker.set_data([x], [y])
    line_spec.set_data(lam_arr[valid], spec[valid])

    if _fill[0] is not None:
        _fill[0].remove()
        _fill[0] = None
    if np.any(valid):
        _fill[0] = ax_spec.fill_between(
            lam_arr[valid],
            (spec - sigma)[valid],
            (spec + sigma)[valid],
            alpha=0.3,
            color="C1",
        )

    ax_spec.set_xlim(lam_arr[0] - 0.02, lam_arr[-1] + 0.02)
    if np.any(valid):
        ylo, yhi = np.nanmin(spec[valid]), np.nanmax(spec[valid])
        margin = max(0.1 * (yhi - ylo), 0.01)
        ax_spec.set_ylim(ylo - margin, yhi + margin)
    ax_spec.set_title(f"Spectrum at ({x}, {y})")
    fig_sv.canvas.draw_idle()


_cid = fig_sv.canvas.mpl_connect("button_press_event", _on_click)
plt.show()